## 5. Feature Engineering: Creating Analytical Columns
Creating strategic indicators and domain-specific ratios to evaluate infrastructure capacity, spending velocity, customer sentiment, and operational connectivity.

*Each dataset is processed in its own cell with the resulting new columns displayed via `.head()`.*


In [ ]:
import pandas as pd
import numpy as np

# Load cleaned datasets directly from ../data/cleaned/
df_district_footfall = pd.read_csv('../data/cleaned/01_district_wise_tourist_footfall_cleaned.csv')
df_nidhi_hospitality = pd.read_csv('../data/cleaned/02_nidhi_hospitality_establishments_cleaned.csv')
df_asi_monuments = pd.read_csv('../data/cleaned/03_asi_protected_monuments_cleaned.csv')
df_inbound_survey = pd.read_csv('../data/cleaned/04_inbound_tourist_survey_cleaned.csv')
df_airport_traffic = pd.read_csv('../data/cleaned/05_airport_passenger_traffic_cleaned.csv')
print('Cleaned datasets loaded for Feature Engineering.')


### 5.1 Feature Engineering: `df_district_footfall`


In [122]:
# Foreign tourist percentage out of total footfall
df_district_footfall['Foreign_Visitor_Share_Pct'] = (
    (df_district_footfall['Foreign_Visitors'] / df_district_footfall['Total_Visitors']) * 100
).round(2)

# Hotel supply pressure per 100,000 visitors
df_district_footfall['Hotels_Per_100k_Visitors'] = (
    (df_district_footfall['Registered_Hotels_Count'] / df_district_footfall['Total_Visitors']) * 100000
).round(2)

# Grievances / complaints per 100,000 visitors
df_district_footfall['Complaints_Per_100k_Visitors'] = (
    (df_district_footfall['Reported_Tourist_Complaints'] / df_district_footfall['Total_Visitors']) * 100000
).round(2)

# Total estimated economic bed-days (Visitors * Stay Duration)
df_district_footfall['Estimated_Visitor_Days'] = (
    df_district_footfall['Total_Visitors'] * df_district_footfall['Avg_Stay_Duration_Days']
).round(0).astype('int64')

print('Features added to df_district_footfall. Verification:')
display(df_district_footfall[['District', 'State_UT', 'Year', 'Foreign_Visitor_Share_Pct', 'Hotels_Per_100k_Visitors', 'Complaints_Per_100k_Visitors', 'Estimated_Visitor_Days']].head(5))


Features added to df_district_footfall. Verification:


,District,State_UT,Year,Foreign_Visitor_Share_Pct,Hotels_Per_100k_Visitors,Complaints_Per_100k_Visitors,Estimated_Visitor_Days
0,Agra,Uttar Pradesh,2022,2.2,18.38,0.09,7855286
1,Agra,Uttar Pradesh,2023,2.2,16.08,1.36,3242212
2,Agra,Uttar Pradesh,2024,2.2,16.99,1.14,9679460
3,Varanasi,Uttar Pradesh,2022,3.19,9.32,1.04,10577660
4,Varanasi,Uttar Pradesh,2023,0.0,8.26,0.0,6589899


### 5.2 Feature Engineering: `df_nidhi_hospitality`


In [123]:
# Room tariff spread (premium price room vs standard room)
df_nidhi_hospitality['Tariff_Spread_INR'] = (
    df_nidhi_hospitality['Room_Tariff_Max_INR'] - df_nidhi_hospitality['Room_Tariff_Min_INR']
)

# Market tier classification based on entry tariff
df_nidhi_hospitality['Tariff_Tier'] = pd.cut(
    df_nidhi_hospitality['Room_Tariff_Min_INR'],
    bins=[-np.inf, 2000, 5000, 10000, np.inf],
    labels=['Budget (<2K)', 'Midscale (2K-5K)', 'Upscale (5K-10K)', 'Luxury (>10K)']
)

# Total estimated guest capacity (assuming 2 persons/room)
df_nidhi_hospitality['Estimated_Guest_Capacity'] = df_nidhi_hospitality['Total_Rooms'] * 2

print('Features added to df_nidhi_hospitality. Verification:')
display(df_nidhi_hospitality[['Establishment_Name', 'Room_Tariff_Min_INR', 'Room_Tariff_Max_INR', 'Tariff_Spread_INR', 'Tariff_Tier', 'Estimated_Guest_Capacity']].head(5))


Features added to df_nidhi_hospitality. Verification:


,Establishment_Name,Room_Tariff_Min_INR,Room_Tariff_Max_INR,Tariff_Spread_INR,Tariff_Tier,Estimated_Guest_Capacity
0,Heritage Chandigarh Suites,2500,2500,0,Midscale (2K-5K),8
1,Mountain Khurda Boutique Stay,2500,6500,4000,Midscale (2K-5K),14
2,Mountain Bhagalpur Palace,7500,11500,4000,Upscale (5K-10K),156
3,Heritage South Andaman House,800,3300,2500,Budget (<2K),26
4,Green Udham Singh Nagar Court,5000,7000,2000,Midscale (2K-5K),22


### 5.3 Feature Engineering: `df_asi_monuments`


In [124]:
# Foreign vs Domestic entry fee multiple
df_asi_monuments['Foreign_Fee_Multiple'] = np.where(
    df_asi_monuments['Entry_Fee_Domestic_INR'] > 0,
    (df_asi_monuments['Entry_Fee_Foreign_INR'] / df_asi_monuments['Entry_Fee_Domestic_INR']).round(1),
    0.0
)

# Monument footfall scale for 2026
df_asi_monuments['Footfall_Scale_2026'] = pd.cut(
    df_asi_monuments['Estimated_Annual_Footfall_2026'],
    bins=[-np.inf, 50000, 250000, 1000000, np.inf],
    labels=['Niche (<50K)', 'Moderate (50K-250K)', 'Major (250K-1M)', 'Mega-Site (>1M)']
)

print('Features added to df_asi_monuments. Verification:')
display(df_asi_monuments[['Monument_Name', 'Architectural_Type', 'Entry_Fee_Domestic_INR', 'Entry_Fee_Foreign_INR', 'Foreign_Fee_Multiple', 'Estimated_Annual_Footfall_2026', 'Footfall_Scale_2026']].head(5))


Features added to df_asi_monuments. Verification:


,Monument_Name,Architectural_Type,Entry_Fee_Domestic_INR,Entry_Fee_Foreign_INR,Foreign_Fee_Multiple,Estimated_Annual_Footfall_2026,Footfall_Scale_2026
0,Taj Mahal,Stupa/Monastery,40,600,15.0,7738564,Mega-Site (>1M)
1,Qutub Minar,Temple/Shrine,25,1100,44.0,7621253,Mega-Site (>1M)
2,Red Fort,Fort/Bastion,50,1100,22.0,3592065,Mega-Site (>1M)
3,Humayun's Tomb,Cave/Rock-Cut,25,550,22.0,1391133,Mega-Site (>1M)
4,Agra Fort,Stupa/Monastery,35,600,17.1,2329816,Mega-Site (>1M)


### 5.4 Feature Engineering: `df_inbound_survey`


In [125]:
# Spending velocity: Average expenditure per day stayed
df_inbound_survey['Spend_Per_Day_USD'] = (
    df_inbound_survey['Estimated_Total_Spend_USD'] / df_inbound_survey['Total_Nights_Stayed']
).round(2)

# Stay duration length tier
df_inbound_survey['Stay_Length_Tier'] = pd.cut(
    df_inbound_survey['Total_Nights_Stayed'],
    bins=[0, 7, 21, 45, np.inf],
    labels=['Short (1-7d)', 'Medium (8-21d)', 'Long (22-45d)', 'Extended (>45d)']
)

# Net Promoter sentiment classification
df_inbound_survey['Promoter_Class'] = np.where(
    (df_inbound_survey['Overall_Experience_Score_1to10'] >= 8) & (df_inbound_survey['Would_Revisit_India'] == 'Yes'),
    'Promoter',
    np.where(df_inbound_survey['Overall_Experience_Score_1to10'] <= 5, 'Detractor', 'Passive')
)

print('Features added to df_inbound_survey. Verification:')
display(df_inbound_survey[['Response_ID', 'Tourist_Nationality', 'Total_Nights_Stayed', 'Estimated_Total_Spend_USD', 'Spend_Per_Day_USD', 'Stay_Length_Tier', 'Promoter_Class']].head(5))


Features added to df_inbound_survey. Verification:


,Response_ID,Tourist_Nationality,Total_Nights_Stayed,Estimated_Total_Spend_USD,Spend_Per_Day_USD,Stay_Length_Tier,Promoter_Class
0,SURV-2024-00001,USA,34,7628.0,224.35,Long (22-45d),Passive
1,SURV-2024-00002,Japan,14,3987.0,284.79,Medium (8-21d),Passive
2,SURV-2024-00003,UK,37,9184.0,248.22,Long (22-45d),Passive
3,SURV-2024-00004,Nepal,25,2260.0,90.40,Long (22-45d),Passive
4,SURV-2024-00005,UK,4,615.0,153.75,Short (1-7d),Detractor


### 5.5 Feature Engineering: `df_airport_traffic`


In [126]:
# Total combined passenger volume (Domestic + International, Arrivals + Departures)
df_airport_traffic['Total_Passengers'] = (
    df_airport_traffic['Domestic_Passengers_Arrival'] + df_airport_traffic['Domestic_Passengers_Departure'] +
    df_airport_traffic['International_Passengers_Arrival'] + df_airport_traffic['International_Passengers_Departure']
)

# International passenger traffic share percentage
df_airport_traffic['International_Share_Pct'] = (
    ((df_airport_traffic['International_Passengers_Arrival'] + df_airport_traffic['International_Passengers_Departure'])
     / df_airport_traffic['Total_Passengers']) * 100
).round(2)

# Flight load: Average passengers transported per aircraft movement
df_airport_traffic['Passengers_Per_Flight_Movement'] = (
    df_airport_traffic['Total_Passengers'] / df_airport_traffic['Total_Aircraft_Movements']
).round(1)

print('Features added to df_airport_traffic. Verification:')
display(df_airport_traffic[['Airport_IATA', 'Year', 'Month', 'Total_Passengers', 'International_Share_Pct', 'Passengers_Per_Flight_Movement']].head(5))


Features added to df_airport_traffic. Verification:


,Airport_IATA,Year,Month,Total_Passengers,International_Share_Pct,Passengers_Per_Flight_Movement
0,DEL,2022,January,8095027,29.99,146.7
1,DEL,2022,February,8116672,27.84,132.4
2,DEL,2022,March,7379977,25.37,136.1
3,DEL,2022,April,6058721,26.12,134.2
4,DEL,2022,May,5769553,27.36,132.5
